# 01. Carga y limpieza

Carga los JSON originales, elimina registros marcados como borrados, normaliza los campos comunes y guarda la base ejecutada en `data/processed/base.parquet`.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
PLOTS_DIR = PROJECT_ROOT / "outputs" / "plots"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"

for directory in [DATA_PROCESSED_DIR, MODELS_DIR, PLOTS_DIR, METRICS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
from IPython.display import display

from src.data_loader import load_json_files
from src.preprocessing import clean_data, create_base_dataframe


In [ ]:
preventivo_raw, correctivo_raw = load_json_files(
    DATA_RAW_DIR / "preventivos.json",
    DATA_RAW_DIR / "correctivos.json",
)

clean_df = clean_data(preventivo_raw, correctivo_raw, empresa_id="RBUS")
base_df = create_base_dataframe(clean_df, executed_only=True)
base_path = DATA_PROCESSED_DIR / "base.parquet"
base_df.to_parquet(base_path, index=False)

print("Cantidad registros preventivo (raw):", len(preventivo_raw))
print("Cantidad registros correctivo (raw):", len(correctivo_raw))
print("Total registros ejecutados en base:", len(base_df))
print("Tipos de revisión ejecutados:")
print(base_df["tipo_revision"].value_counts(dropna=False))
print("Rango temporal:", base_df["fecha_evento"].min(), "→", base_df["fecha_evento"].max())
print("Buses únicos:", base_df["placa_patente"].nunique())
print("Parquet guardado en:", base_path)

preview_columns = [
    "firebase_id",
    "tipo_revision",
    "placa_patente",
    "fecha_evento",
    "pauta_proyectada",
    "causa_origen",
    "sistema_componente",
    "taller_planta",
]
preview_columns = [column for column in preview_columns if column in base_df.columns]
display(base_df[preview_columns].head())
